# 02 · Active-load common-source amplifier: gm/id design with novaad + ngspice verification

A single-stage amplifier: NMOS common-source `M1` loaded by a PMOS current source `M2`, driving a capacitive load `CL`.

```
        VDD
         │
VBP ──┤ M2  (sg13_lv_pmos, current-source load)
         │
         ├──── out ──┬── CL
         │           │
 in ──┤ M1          ═╧═
         │  (sg13_lv_nmos)
        GND
```

Design flow (gm/id method):
1. **GBW** fixes `gm1 = 2π·GBW·CL`.
2. The chosen **gm/id of M1** sets the inversion level, so `Ids = gm1 / (gm/id)`. novaad returns `W1` and `Vgs1`.
3. **M2** carries the same `Ids` at its own gm/id. novaad returns `W2` and `Vsg2`, which is the load bias `VBP = VDD − Vsg2`.
4. novaad's electric model predicts gain `gm1/(gds1+gds2)` and the output capacitance, before any simulation.
5. ngspice confirms it with **DC operating point, AC, transient and noise**.

Run `01_pdk_ngspice_lut_extraction.ipynb` first. It installs the PDK and writes the LUTs used here.

In [1]:
# ── Design specifications: edit these ───────────────────────────────────────
VDD = 1.2              # supply [V]: IHP SG13G2 1.2 V core devices
CL = 0.5e-12             # load capacitance [F]
GBW_MIN = 470e6         # gain-bandwidth product [Hz]
GAIN_MIN_DB = 26       # DC gain [dB]
INPUT_NOISE_MAX = 40e-6  # input-referred noise [Vrms]: output noise (NOISE_FMIN..GBW) / DC gain
NOISE_FMIN = 1.0       # lower bound of the noise integration [Hz]
POWER_MAX = 200e-6     # power budget [W]
VOUT_DC = VDD / 2      # output DC operating point [V]

# ── Design choices (the knobs of the gm/id method) ──────────────────────────
GBW_MARGIN = 1.2       # design for GBW_MIN·margin: covers the parasitic output capacitance
L_N = 0.5e-6           # M1 channel length [m]
L_P = 0.5e-6             # M2 channel length [m]
GMID_N = 16            # M1 gm/id [1/V]: moderate inversion, good gm per current
GMID_P = 8             # M2 gm/id [1/V]: stronger inversion, low gm (noise) and good matching

# ── Transient test signals ──────────────────────────────────────────────────
TRAN_FREQ = 20e3       # sine frequency [Hz]: keep it well below the -3 dB corner
TRAN_AMP = 1e-3        # sine amplitude [V]
RAMP_LOW = 0.0         # input ramp start [V]: the ramp spans the full supply
RAMP_HIGH = VDD        # input ramp end [V]
RAMP_TIME = 200e-6     # ramp duration [s]: slow compared to the output time constant
STEP_LOW = 0.0         # input pulse low level [V]: the step spans the full supply
STEP_HIGH = VDD        # input pulse high level [V]
STEP_TIME = 5e-6       # pulse window [s]: the input steps up at 10 % and back down at 50 %
NCH_LUT = "luts/sg13_lv_nch.csv"
PCH_LUT = "luts/sg13_lv_pch.csv"
REF_WIDTH = 10e-6      # width the LUTs were extracted at

In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from math import ceil, log10, pi

import numpy as np
from pandas import DataFrame
from bokeh.io import output_notebook, show
from bokeh.layouts import gridplot
from bokeh.plotting import figure

import spice
from novaad import Device, DeviceSizingSpecification

output_notebook(hide_banner=True)
nch = Device(NCH_LUT, device_type="nch", ref_width=REF_WIDTH)
pch = Device(PCH_LUT, device_type="pch", ref_width=REF_WIDTH)

## 1. Size the devices with novaad

In [3]:
gm1 = 2 * pi * GBW_MIN * GBW_MARGIN * CL

dcop_n, size_n = nch.sizing(
    DeviceSizingSpecification(vds=VOUT_DC, vsb=0.0, lch=L_N, gmoverid=GMID_N, gm=gm1),
    return_dcop=True,
)
IDS = dcop_n.ids[0]
dcop_p, size_p = pch.sizing(
    DeviceSizingSpecification(vds=VDD - VOUT_DC, vsb=0.0, lch=L_P, gmoverid=GMID_P, ids=IDS),
    return_dcop=True,
)
W_N, W_P = size_n.wch[0], size_p.wch[0]
VGS_N, VSG_P = dcop_n.vgs[0], dcop_p.vgs[0]

em_n = nch.electric_model(dcop_n, size_n)
em_p = pch.electric_model(dcop_p, size_p)
gain_pred = em_n.gm[0] / (em_n.gds[0] + em_p.gds[0])
c_out = CL + em_n.cdb[0] + em_p.cdb[0] + em_n.cgd[0] + em_p.cgd[0]
gbw_pred = em_n.gm[0] / (2 * pi * c_out)

DataFrame({
    "M1 (nch)": [W_N * 1e6, L_N * 1e6, VGS_N, IDS * 1e6, em_n.gm[0] * 1e6, em_n.gds[0] * 1e6],
    "M2 (pch)": [W_P * 1e6, L_P * 1e6, VSG_P, IDS * 1e6, em_p.gm[0] * 1e6, em_p.gds[0] * 1e6],
}, index=["W [um]", "L [um]", "|Vgs| [V]", "Ids [uA]", "gm [uS]", "gds [uS]"]).round(3)

,M1 (nch),M2 (pch)
W [um],40.701,26.127
L [um],0.500,0.500
|Vgs| [V],0.312,0.587
Ids [uA],108.649,108.649
gm [uS],1748.491,876.899
gds [uS],63.766,12.476


In [4]:
print(f"predicted DC gain : {20 * log10(gain_pred):.1f} dB   (spec >= {GAIN_MIN_DB} dB)")
print(f"predicted GBW     : {gbw_pred / 1e6:.1f} MHz (spec >= {GBW_MIN / 1e6:.0f} MHz, Cout = {c_out * 1e15:.0f} fF)")
print(f"predicted power   : {VDD * IDS * 1e6:.1f} uW  (spec <= {POWER_MAX * 1e6:.0f} uW)")

predicted DC gain : 27.2 dB   (spec >= 26 dB)
predicted GBW     : 498.9 MHz (spec >= 470 MHz, Cout = 558 fF)
predicted power   : 130.4 uW  (spec <= 200 uW)


## 2. Testbench

The netlist is built by two generators:
- **`dut(d)`** writes the amplifier as a `.subckt` macro with pins `avdd avss in out vbp`.
- **`testbench(d, control, vin, vin_source)`** adds the supply, the input source and the load bias around it, instantiates the macro as `XDUT`, and appends the `.control` block.

The DUT's devices are therefore addressed hierarchically, for example `XDUT.XM1` in the DC table. Each transistor is an instance of the PDK subcircuit, split into `ng` fingers no wider than `REF_WIDTH`, matching how the LUT was characterized. The PMOS gate is driven by the ideal bias `VBP = VDD − Vsg2` predicted by novaad.


In [5]:
def fet(name, d, g, s, b, model, w, l):
    ng = max(1, ceil(w / REF_WIDTH - 1e-9))   # REF_WIDTH-wide fingers, like the LUT
    return f"X{name} {d} {g} {s} {b} {model} w={w:.6g} l={l:.6g} ng={ng}"


def dut(d: dict, name: str = "cs_amp") -> str:
    # pins: supplies, input, output, load gate bias
    return f'''.subckt {name} avdd avss in out vbp
* active-load common-source amplifier
{fet("M1", "out", "in", "avss", "avss", spice.NMOS, d["w_n"], d["l_n"])}
{fet("M2", "out", "vbp", "avdd", "avdd", spice.PMOS, d["w_p"], d["l_p"])}
.ends {name}'''


# `vin` is the input DC bias; `vin_source` overrides the input waveform (sin/pwl/pulse) for the transient runs
def testbench(d: dict, control: str, vin: float = 0.0, vin_source: str | None = None) -> str:
    return spice.header() + dut(d) + f'''
VDD vdd 0 {VDD}
VBP vbp 0 {d["vbp"]}
VIN in 0 {vin_source or f"dc {vin} ac 1"}
XDUT vdd 0 in out vbp cs_amp
CL out 0 {CL}
.control
set wr_singlescale
set wr_vecnames
{control}
.endc
.end
'''


d = dict(w_n=W_N, l_n=L_N, w_p=W_P, l_p=L_P, vbp=VDD - VSG_P)
print(testbench(d, "op", VGS_N))

* novaad testbench
.lib /Users/gfdasdias/.ciel/ihp-sg13g2/libs.tech/ngspice/models/cornerMOSlv.lib mos_tt
.temp 27
.option savecurrents
.subckt cs_amp avdd avss in out vbp
* active-load common-source amplifier
XM1 out in avss avss sg13_lv_nmos w=4.07006e-05 l=5e-07 ng=5
XM2 out vbp avdd avdd sg13_lv_pmos w=2.61266e-05 l=5e-07 ng=3
.ends cs_amp
VDD vdd 0 1.2
VBP vbp 0 0.6125
VIN in 0 dc 0.3125 ac 1
XDUT vdd 0 in out vbp cs_amp
CL out 0 5e-13
.control
set wr_singlescale
set wr_vecnames
op
.endc
.end



## 3. DC: find the input bias

The output of a high-gain stage is very sensitive to its input bias: one LSB of model error moves `vout` to a rail. The `vin` that puts `vout` at `VOUT_DC` is therefore found from a fine DC transfer sweep. A real circuit would set it with feedback.

In [6]:
dc = spice.read(spice.run(testbench(d, f"dc VIN 0 {VDD} 0.2m\nwrdata dc.txt v(out)")) / "dc.txt")
VIN_DC = float(np.interp(VOUT_DC, dc["v(out)"][::-1], dc["v-sweep"][::-1]))   # vout falls with vin
print(f"input bias for vout = {VOUT_DC:.3f} V: vin = {VIN_DC * 1e3:.1f} mV (novaad predicted Vgs = {VGS_N * 1e3:.0f} mV)")

p = figure(title="DC transfer", x_axis_label="vin [V]", y_axis_label="vout [V]", width=500, height=300)
p.line(dc["v-sweep"], dc["v(out)"], line_width=2)
p.scatter([VIN_DC], [VOUT_DC], size=8, color="red")
show(p)

input bias for vout = 0.600 V: vin = 318.5 mV (novaad predicted Vgs = 312 mV)


## 4. DC operating point

One `op` run saves every node voltage and every element current. `utils.dc_tables` uses InSpice's parse of the netlist to map each element's terminals to nets, and builds the tables below. MOSFET rows also show the PSP103 bias quantities and the operating region.

In [7]:
from IPython.display import display
from utils import dc_tables, dc_vectors

bench = testbench(d, "op", VIN_DC)
op = spice.read(spice.run(testbench(d, f"op\nwrdata op.txt {dc_vectors(bench)}", VIN_DC)) / "op.txt")
devices, nodes = dc_tables(bench, op)
display(devices.round(4).fillna(""))
nodes.round(4)

,type,nets,I [uA],V(+) [V],V(-) [V],V(d) [V],V(g) [V],V(s) [V],V(b) [V],vgs [V],vds [V],vsb [V],vth [V],vdss [V],gm [uS],gds [uS],region
element,,,,,,,,,,,,,,,,,
VDD,V,+=vdd -=0,-108.759,1.2,0.0,,,,,,,,,,,,
VBP,V,+=vbp -=0,0.000,0.6125,0.0,,,,,,,,,,,,
VIN,V,+=in -=0,-0.000,0.3185,0.0,,,,,,,,,,,,
XDUT.XM1,sg13_lv_nmos,d=out g=in s=0 b=0,108.759,,,0.6,0.3185,0.0,0.0,0.3185,0.6,0.0,0.2599,0.134,1753.6406,64.7202,saturation
XDUT.XM2,sg13_lv_pmos,d=out g=vbp s=vdd b=vdd,108.759,,,0.6,0.6125,1.2,1.2,0.5875,0.6,-0.0,0.365,0.2181,880.044,12.4764,saturation
CL,C,+=out -=0,0.000,0.6,0.0,,,,,,,,,,,,


,V [V]
node,
in,0.3185
out,0.6000
vbp,0.6125
vdd,1.2000


## 5. AC: Bode plot

In [8]:
ac = spice.read(spice.run(testbench(d, "ac dec 50 1k 10G\nwrdata ac.txt vdb(out) vp(out)", VIN_DC)) / "ac.txt")
f = ac["frequency"].to_numpy()
mag = ac["vdb(out)"].to_numpy()
phase = np.degrees(np.unwrap(ac["vp(out)"].to_numpy()))
gain_db, f3db, gbw = spice.bandwidth(f, mag)          # GBW = A0·f-3dB
print(f"DC gain {gain_db:.2f} dB, f-3dB {f3db / 1e3:.1f} kHz, GBW {gbw / 1e6:.2f} MHz, "
      f"phase at GBW {np.interp(gbw, f, phase):.1f} deg")

p_mag = figure(title="Magnitude", x_axis_type="log", x_axis_label="f [Hz]", y_axis_label="|vout/vin| [dB]", width=900, height=280)
p_mag.line(f, mag, line_width=2)
p_mag.scatter([f3db], [gain_db - 3], size=8, color="red")
p_mag.line([f3db, gbw], [gain_db, 0], line_dash="dashed", color="gray", legend_label="single-pole asymptote to GBW")
p_ph = figure(title="Phase", x_axis_type="log", x_axis_label="f [Hz]", y_axis_label="phase [deg]", width=900, height=280, x_range=p_mag.x_range)
p_ph.line(f, phase, line_width=2)
show(gridplot([[p_mag], [p_ph]]))

DC gain 27.13 dB, f-3dB 22590.1 kHz, GBW 513.17 MHz, phase at GBW 87.8 deg


## 6. Transient: sine, ramp and step

Three separate runs, each with its own input source on `VIN` around the bias point `VIN_DC`:
- **Sine:** small-signal gain at `TRAN_FREQ`.
- **Ramp:** a slow ramp from `RAMP_LOW` to `RAMP_HIGH` (the full supply). The output traces the whole large-signal transfer curve, rail to rail.
- **Step:** a pulse from `STEP_LOW` to `STEP_HIGH` and back (the full supply). These are large-signal edges, so the output **slews**:
  - When the input falls, M1 turns off and only the PMOS current charges the output, so the rising output slew rate should be about `Ids/Cout`. `Cout` is `CL` plus the drain capacitances predicted by novaad.
  - Slew rates are measured as the output slope between 40 % and 60 % of each transition (`spice.slew_rate`).
  - When the input rises, M1 is driven with the full supply on its gate and pulls the output down much faster.

Input and output share the same voltage axis, so the gain is visible directly as the ratio of the two swings.

In [9]:
from utils import tran_plot


def tran(source: str, tstop: float):
    wd = spice.run(testbench(d, f"tran {tstop / 2000} {tstop}\nwrdata tran.txt v(in) v(out)", VIN_DC, source))
    return spice.read(wd / "tran.txt")


sine = tran(f"dc {VIN_DC} sin({VIN_DC} {TRAN_AMP} {TRAN_FREQ})", 6 / TRAN_FREQ)
ramp = tran(f"pwl(0 {RAMP_LOW} {RAMP_TIME} {RAMP_HIGH})", RAMP_TIME)
t_up, t_down = STEP_TIME / 10, STEP_TIME / 2
step = tran(f"pulse({STEP_LOW} {STEP_HIGH} {t_up} 1n 1n {t_down - t_up} 1)", STEP_TIME)

# sine: amplitude over the last period, after the start-up transient
last = sine["time"] >= 5 / TRAN_FREQ
gain_tran_db = 20 * log10((sine["v(out)"][last].max() - sine["v(out)"][last].min()) / 2 / TRAN_AMP)
# pulse: 10-90 % slew rates of the falling (input up) and rising (input down) output edges
sr_fall = spice.slew_rate(step["time"], step["v(out)"], t_up, t_down)
sr_rise = spice.slew_rate(step["time"], step["v(out)"], t_down, STEP_TIME)
print(f"sine gain {gain_tran_db:.2f} dB | ramp output {ramp['v(out)'].min():.2f} .. {ramp['v(out)'].max():.2f} V | "
      f"slew rate: rising {sr_rise / 1e6:.1f} V/us (Ids/Cout = {IDS / c_out / 1e6:.1f}), falling {sr_fall / 1e6:.1f} V/us")

show(gridplot([
    [tran_plot(f"Sine: {TRAN_AMP * 1e3:g} mV @ {TRAN_FREQ / 1e3:g} kHz", sine["time"], sine["v(in)"], sine["v(out)"])],
    [tran_plot(f"Ramp: {RAMP_LOW:g} V to {RAMP_HIGH:g} V in {RAMP_TIME * 1e6:g} us", ramp["time"], ramp["v(in)"], ramp["v(out)"])],
    [tran_plot(f"Step: {STEP_LOW:g} V to {STEP_HIGH:g} V and back", step["time"], step["v(in)"], step["v(out)"])],
]))

sine gain 27.13 dB | ramp output 0.00 .. 1.20 V | slew rate: rising 104.6 V/us (Ids/Cout = 194.8), falling 1086.8 V/us


## 7. Noise

The output noise is integrated from `NOISE_FMIN` up to the GBW and referred to the input by the DC gain: `vn,in = vn,out / A0`. It must stay below `INPUT_NOISE_MAX`. `set sqrnoise` makes ngspice report V²/Hz and V².

The novaad estimate is **thermal noise only**: `S_in = 4kT·γ·(1 + gm2/gm1)/gm1` over the noise bandwidth `π/2·f−3dB`, with γ = 2/3. The gap to ngspice is mostly flicker noise, which dominates at low frequency.

In [10]:
K_BOLTZMANN, T_KELVIN, GAMMA = 1.380649e-23, 300.15, 2 / 3

wd = spice.run(testbench(d, f'''set sqrnoise
noise v(out) VIN dec 20 {NOISE_FMIN} {gbw}
setplot noise1
wrdata noise_spec.txt inoise_spectrum onoise_spectrum
setplot noise2
wrdata noise_tot.txt onoise_total''', VIN_DC))
noise = spice.read(wd / "noise_spec.txt")
vn_out = float(np.sqrt(spice.read(wd / "noise_tot.txt")["onoise_total"][0]))
vn_in = vn_out / 10 ** (gain_db / 20)

s_in = 4 * K_BOLTZMANN * T_KELVIN * GAMMA * (1 + em_p.gm[0] / em_n.gm[0]) / em_n.gm[0]
vn_in_pred = float(np.sqrt(s_in * pi / 2 * gbw_pred / gain_pred))
print(f"output noise {vn_out * 1e6:.1f} uVrms -> input-referred {vn_in * 1e6:.2f} uVrms "
      f"(spec <= {INPUT_NOISE_MAX * 1e6:g}, novaad thermal estimate {vn_in_pred * 1e6:.2f})")

p_noise = figure(title="Noise density", x_axis_type="log", y_axis_type="log", x_axis_label="f [Hz]",
                 y_axis_label="nV/√Hz", width=900, height=300)
p_noise.line(noise["frequency"], np.sqrt(noise["inoise_spectrum"]) * 1e9, line_width=2, legend_label="input-referred")
p_noise.line(noise["frequency"], np.sqrt(noise["onoise_spectrum"]) * 1e9, line_width=2, color="orange", legend_label="output")
show(p_noise)

output noise 676.6 uVrms -> input-referred 29.78 uVrms (spec <= 40, novaad thermal estimate 18.01)


## 8. Did the design meet the specs?

In [11]:
from utils import footprint

power = VDD * abs(op["i(vdd)"][0])
gate_area = W_N * L_N + W_P * L_P   # active gate area of M1 + M2, before layout overheads
# single-pole roll-off of the predicted gain at the transient test frequency
gain_pred_tran_db = 20 * log10(gain_pred) - 10 * log10(1 + (TRAN_FREQ * gain_pred / gbw_pred) ** 2)
sat = bool(devices["region"].dropna().eq("saturation").all())   # MOSFET rows only
report = DataFrame([
    ("DC gain [dB]", f">= {GAIN_MIN_DB}", 20 * log10(gain_pred), gain_db, gain_db >= GAIN_MIN_DB),
    # the sine sits on the pole's roll-off: compare it with the AC gain at the same frequency
    ("Transient gain [dB]", f"= AC gain at {TRAN_FREQ / 1e3:g} kHz: {np.interp(TRAN_FREQ, f, mag):.2f}",
     gain_pred_tran_db, gain_tran_db, abs(gain_tran_db - np.interp(TRAN_FREQ, f, mag)) < 0.2),
    ("GBW [MHz]", f">= {GBW_MIN / 1e6:g}", gbw_pred / 1e6, gbw / 1e6, gbw >= GBW_MIN),
    ("-3 dB bandwidth [kHz]", "", gbw_pred / gain_pred / 1e3, f3db / 1e3, True),
    ("Slew rate, output rising [V/us]", "", IDS / c_out / 1e6, sr_rise / 1e6, True),
    ("Slew rate, output falling [V/us]", "", "", sr_fall / 1e6, True),
    ("Input-referred noise [uVrms]", f"<= {INPUT_NOISE_MAX * 1e6:g}", vn_in_pred * 1e6, vn_in * 1e6, vn_in <= INPUT_NOISE_MAX),
    ("Power [uW]", f"<= {POWER_MAX * 1e6:g}", VDD * IDS * 1e6, power * 1e6, power <= POWER_MAX),
    ("Vout DC [V]", f"= {VOUT_DC:.2f}", VOUT_DC, op["v(out)"][0], abs(op["v(out)"][0] - VOUT_DC) < 10e-3),
    ("All devices saturated", "yes", "", "yes" if sat else "no", sat),
    ("Gate area [um²]", "", gate_area * 1e12, "", True),
    ("Gate area, equivalent square", "", footprint(gate_area), "", True),
], columns=["metric", "spec", "novaad prediction", "ngspice", "pass"]).set_index("metric")
report.style.format(precision=2).map(lambda v: "color: green" if v is True else ("color: red" if v is False else ""), subset=["pass"])

,spec,novaad prediction,ngspice,pass
metric,,,,
DC gain [dB],>= 26,27.21,27.13,True
Transient gain [dB],= AC gain at 20 kHz: 27.13,27.21,27.13,True
GBW [MHz],>= 470,498.92,513.17,True
-3 dB bandwidth [kHz],,21754.84,22590.10,True
"Slew rate, output rising [V/us]",,194.79,104.64,True
"Slew rate, output falling [V/us]",,,1086.76,True
Input-referred noise [uVrms],<= 40,18.01,29.78,True
Power [uW],<= 200,130.38,130.51,True
Vout DC [V],= 0.60,0.60,0.60,True
